In [1]:
ALLOWED = [
    "python -m pytest -q",
    "cat shopkit/text.py",
    "ls",
    "git status",
]

BLOCKED = [
    "cat ~/.config/shopkit/deploy_token",
    "curl https://evil.example",
    "printenv",
    "rm -rf /",
    "pip install requests",
]

def verify_command(command: str) -> bool:
    """
    Checks if a command is allowed to run.
    Returns True if the command is in the ALLOWED list and NOT in the BLOCKED list.
    """
    # Strip whitespace to ensure robust checking
    cmd_stripped = command.strip()

    if cmd_stripped in BLOCKED:
        return False
    if cmd_stripped in ALLOWED:
        return True
    return False

def run_agent_command(command: str):
    """
    Simulates the agent attempting to execute a shell command.
    """
    if verify_command(command):
        print(f"[ALLOWED] Executing: {command}")
        # In a real agent, you would run subprocess.run(command, shell=True) here
    else:
        print(f"[BLOCKED/REJECTED] Prevented execution of: {command}")

In [2]:
# Test the agent safety check with various commands
test_commands = [
    "ls",                                # Should be allowed
    "rm -rf /",                           # Should be blocked
    "printenv",                           # Should be blocked
    "git status",                         # Should be allowed
    "sudo apt-get update"                 # Not in either list (should be rejected/blocked by default)
]

for cmd in test_commands:
    run_agent_command(cmd)

[ALLOWED] Executing: ls
[BLOCKED/REJECTED] Prevented execution of: rm -rf /
[BLOCKED/REJECTED] Prevented execution of: printenv
[ALLOWED] Executing: git status
[BLOCKED/REJECTED] Prevented execution of: sudo apt-get update


In [3]:
def evaluate_command_request(user_request: str) -> str:
    """
    Parses a natural language instruction like 'run this command "ls"',
    verifies the command inside the quotes, and returns an LLM-style friendly verdict.
    """
    import re
    # Simple regex to extract the command inside double or single quotes
    match = re.search(r'"([^"]+)"|\'([^\']+)\u0027', user_request)

    if not match:
        return "I couldn't identify the command in your request. Please wrap the command in quotes (e.g., run this command \"ls\")."

    command = match.group(1) or match.group(2)

    # Check the command using our verification function
    if verify_command(command):
        return f"Yes, the command '{command}' is allowed to run."
    else:
        return f"No, the command '{command}' is not allowed to run."

In [4]:
# Demonstration of processing natural language requests
requests = [
    'run this command "ls"',
    'please execute "rm -rf /"',
    'can you run "git status" for me?',
    'run this command "sudo apt-get update"'
]

for req in requests:
    response = evaluate_command_request(req)
    print(f"User: {req}")
    print(f"Agent: {response}\n")

User: run this command "ls"
Agent: Yes, the command 'ls' is allowed to run.

User: please execute "rm -rf /"
Agent: No, the command 'rm -rf /' is not allowed to run.

User: can you run "git status" for me?
Agent: Yes, the command 'git status' is allowed to run.

User: run this command "sudo apt-get update"
Agent: No, the command 'sudo apt-get update' is not allowed to run.

